In [0]:
# ============================================================
#  RAW → SILVER: ENTREGAS
# ============================================================

STORAGE_ACCOUNT  = "stdatacolake"
CONTAINER_RAW    = "raw-zone"
CONTAINER_SILVER = "silver-zone"

RAW_PATH    = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/GPS"
SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/entregas"

In [0]:
# ── LEER CSV DESDE RAW ────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

schema_entregas = StructType([
    StructField("id_entrega",           IntegerType(), nullable=False),
    StructField("id_vehiculo",          IntegerType(), nullable=True),
    StructField("id_ruta",              IntegerType(), nullable=True),
    StructField("fecha_entrega",        TimestampType(),nullable=True),
    StructField("hora_salida",          StringType(),  nullable=True),
    StructField("hora_llegada",         StringType(),  nullable=True),
    StructField("tiempo_entrega_min",   IntegerType(), nullable=True),
    StructField("consumo_combustible",  DoubleType(),  nullable=True),
    StructField("id_venta",             IntegerType(), nullable=True),
    StructField("cumplimiento",         IntegerType(), nullable=True),
])

df_raw = spark.read \
    .schema(schema_entregas) \
    .option("header", "true") \
    .csv(f"{RAW_PATH}/Entregas.csv")

print(f"✅ Raw leído: {df_raw.count()} filas")
df_raw.show(5)

In [0]:
# ── LIMPIEZA Y TRANSFORMACIONES ───────────────────────────────
from pyspark.sql.functions import col, trim, when, current_timestamp

df_silver = df_raw \
    .filter(col("id_entrega").isNotNull()) \
    .filter(col("id_vehiculo").isNotNull()) \
    .filter(col("tiempo_entrega_min") > 0) \
    .filter(col("consumo_combustible") > 0) \
    .withColumn("cumplimiento_desc",
        when(col("cumplimiento") == 1, "Cumplido")
       .otherwise("No cumplido")
    ) \
    .withColumn("eficiencia_combustible",
        (col("tiempo_entrega_min") / col("consumo_combustible")).cast(DoubleType())
    ) \
    .dropDuplicates(["id_entrega"]) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ Silver listo: {df_silver.count()} filas")
df_silver.show(5)

In [0]:
# ── GUARDAR COMO DELTA EN SILVER ─────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PATH)

print(f"✅ Guardado en Silver: {SILVER_PATH}")